# US County Tariff-Exposure — Data Manipulation (Canada→US retaliatory counter-tariffs)

Mirror of the Canadian `Data_Manipulation_YJ_CSD.ipynb`, reversed: impact on **US counties**
from **Canadian** counter-tariffs, with a **pre-Aug-31 vs post-Sep-1-2025** policy-coverage toggle.

**Pipeline (simpler than the Canadian one — county IS the geography, no DA→ADA rollup, no commute model):**
1. Tariffed HS lists (pre/post) → HS6→NAICS via concordance  → **truncate to NAICS4** (export-data ceiling)
2. State exports-to-Canada ratio per NAICS4 (USA Trade Online)
3. QCEW county employment in tariffed NAICS4 (agglvl 76, private), weighted by state ratio
4. County exposure % → choropleth; overlay 2024 vote margin (MIT Election Lab)

**Depth decision:** USA Trade Online state NAICS exports cap at **4-digit**, so the national join is at NAICS4.
QCEW NAICS4 (agglvl_code 76) is also far less disclosure-suppressed than NAICS6 — a bonus for county coverage.
Use NAICS6 (agglvl 78) only for the hand-built case studies (Detroit-Windsor auto, Sault Ste. Marie steel).


In [15]:
import numpy as np
import pandas as pd
import geopandas as gpd
import os, glob
from pathlib import Path


## Configuration — paths (edit to your local layout)

In [16]:
# ---- INPUT PATHS (edit) ----
TARIFF_PRE   = '../raw/counter_tariffs_pre.csv'      # 1814 lines, effective up to Aug 31 2025
TARIFF_POST  = '../raw/counter_tariffs_post.csv'     # 313 lines, effective Sep 1 2025
CONCORDANCE  = '../raw/C616_HS8toNaics6_concord_202505.csv'
STATE_EXPORT = '../raw/naics_state_export.csv'       # USA Trade Online, state exports to Canada, NAICS
WORLD_EXPORT = '../raw/naics_world_export.csv'         # USA Trade Online, state exports to the World, NAICS
QCEW_DIR     = '../raw/2025.annual.by_area/'         # 4,451 single-area annual CSVs
COUNTY_VOTE  = '../raw/countypres_2000-2024.csv'     # MIT Election Lab

OUT_DIR = '../outputs'
os.makedirs(f'{OUT_DIR}/geojson', exist_ok=True)
os.makedirs(f'{OUT_DIR}/csv', exist_ok=True)

NAICS_DEPTH = 4          # national-map depth (export ceiling). QCEW agglvl_code 76 = county NAICS4 private.
QCEW_AGGLVL = 76         # 74=sector,75=NAICS3,76=NAICS4,77=NAICS5,78=NAICS6
QCEW_OWN    = 5          # 5 = private


# STEP 1 — Tariffed HS lists → NAICS via concordance (pre & post)

Mirrors Canadian cells 5–9, minus the CUSMA-utilisation layer (Canada-specific). We keep BOTH
tariff sets so every downstream metric can be computed for pre and post (the toggle).

In [17]:
def load_tariff(path, label):
    t = pd.read_csv(path, dtype=str)
    # the scraper named the HS column 'tariff_item'; fall back to first col if renamed
    hs_col = 'tariff_item' if 'tariff_item' in t.columns else t.columns[0]
    t['hs6'] = t[hs_col].str.replace(r'\D','',regex=True).str[:6]
    t = t[['hs6']].dropna().drop_duplicates()
    t['period'] = label
    print(f'{label}: {len(t)} unique HS6')
    return t

tar_pre  = load_tariff(TARIFF_PRE,  'pre')
tar_post = load_tariff(TARIFF_POST, 'post')

pre_codes, post_codes = set(tar_pre['hs6']), set(tar_post['hs6'])
print('overlap (post should be subset of pre):', len(pre_codes & post_codes), 'of', len(post_codes))
print('removed Sep 1 (pre not post):', len(pre_codes - post_codes))


pre: 1321 unique HS6
post: 288 unique HS6
overlap (post should be subset of pre): 288 of 288
removed Sep 1 (pre not post): 1033


In [18]:
# Concordance HS6 -> NAICS, truncated to NAICS_DEPTH (4)
conc = pd.read_csv(CONCORDANCE, dtype=str)
# Canadian notebook used columns 'HS8 Code' and 'NAICS 6 Code' — adjust if yours differ
hs8_col   = next(c for c in conc.columns if 'HS8' in c or 'hts' in c.lower() or 'HS' in c)
naics_col = next(c for c in conc.columns if 'NAICS' in c.upper())
print('using concordance cols:', hs8_col, '/', naics_col)

conc['hs6']    = conc[hs8_col].astype(str).str.replace(r'\D','',regex=True).str.zfill(8).str[:6]
conc['naics']  = conc[naics_col].astype(str).str.replace(r'\D','',regex=True).str[:NAICS_DEPTH]
conc = conc[['hs6','naics']].drop_duplicates()
print('concordance rows (HS6->NAICS%d):' % NAICS_DEPTH, len(conc))


using concordance cols: HS8 Code / NAICS 6 Code
concordance rows (HS6->NAICS4): 6628


In [19]:
# Map each tariff set to its NAICS4 set
def hs_to_naics(tar):
    m = tar.merge(conc, on='hs6', how='left')
    miss = m['naics'].isna().mean()
    print(f'  unmapped HS6 share: {miss:.1%}')
    return set(m['naics'].dropna().unique())

naics_pre  = hs_to_naics(tar_pre)
naics_post = hs_to_naics(tar_post)
naics_removed = naics_pre - naics_post
print(f'NAICS4 exposed — pre: {len(naics_pre)}, post: {len(naics_post)}, removed: {len(naics_removed)}')


  unmapped HS6 share: 0.8%
  unmapped HS6 share: 0.0%
NAICS4 exposed — pre: 82, post: 19, removed: 63


# STEP 2 — State export-to-Canada ratio per NAICS4

Mirrors Canadian cells 12–13, but the ratio is **Canada-exports / all-state-exports-of-that-NAICS**.
The export file only has exports TO Canada, so for a true "share of output going to Canada" you'd need
each state's TOTAL (world) exports per NAICS as the denominator.

**TODO (denominator):** if you want the Canada-intensity ratio, also pull state exports to **World**
(USA Trade Online, same NAICS, Country=World) and divide. For v1 we use Canada export *value* directly
as the exposure weight (dollars of Canada-bound exports per NAICS per state), which is defensible as an
absolute-exposure measure even without the ratio.

In [20]:
exp = pd.read_csv(STATE_EXPORT, skiprows=2)
exp = exp.dropna(axis=1, how='all')
exp.columns = ['state','commodity','country','year','export_value']
exp['naics'] = exp['commodity'].str.extract(r'^(\d+)')
exp['export_value'] = (exp['export_value'].astype(str).str.replace(',','',regex=False)
                       .replace('',pd.NA).astype(float))
exp = exp[exp['naics'].str.len()==NAICS_DEPTH].copy()       # NAICS4 rows only
exp = exp[exp['state']!='All States'].copy()                # keep individual states
# state name -> 2-letter not needed yet; QCEW join is via FIPS, we map state below
canada_exp = exp.groupby(['state','naics'], as_index=False)['export_value'].sum()
print('state x naics4 export rows:', len(canada_exp), '| states:', canada_exp['state'].nunique())
canada_exp.head()


state x naics4 export rows: 5097 | states: 54


,state,naics,export_value
0,Alabama,1112,596067.0
1,Alabama,1119,2196319.0
2,Alabama,1123,32396617.0
3,Alabama,1129,280157.0
4,Alabama,1132,9888.0


In [25]:
# Load and process World exports (same structure, different file)
exp_world = pd.read_csv(WORLD_EXPORT, skiprows=2)
exp_world = exp_world.dropna(axis=1, how='all')
exp_world.columns = ['state','commodity','country','year','export_value']
exp_world['naics'] = exp_world['commodity'].str.extract(r'^(\d+)')
exp_world['export_value'] = (exp_world['export_value'].astype(str).str.replace(',','',regex=False)
                             .replace('',pd.NA).astype(float))
exp_world = exp_world[exp_world['naics'].str.len()==NAICS_DEPTH].copy()
exp_world = exp_world[exp_world['state']!='All States'].copy()
world_exp = exp_world.groupby(['state','naics'], as_index=False)['export_value'].sum()

# Merge to calculate Canada share of total exports
exp_ratio = canada_exp.merge(world_exp, on=['state','naics'], how='left', suffixes=('_canada','_world'))
exp_ratio['export_value'] = exp_ratio['export_value_canada']  # keep Canada value
exp_ratio['canada_share'] = np.where(
    exp_ratio['export_value_world'] > 0,
    exp_ratio['export_value_canada'] / exp_ratio['export_value_world'],
    0  # if no world exports, share is 0
)
exp_ratio['export_value_world'] = exp_ratio['export_value_world'].fillna(0)

print('state x naics4 export rows:', len(exp_ratio), '| states:', exp_ratio['state'].nunique())
exp_ratio.head()


canada_exp = exp_ratio.copy()

print('state x naics4 export rows:', len(canada_exp), '| states:', canada_exp['state'].nunique())
canada_exp.head()

state x naics4 export rows: 4885 | states: 50
state x naics4 export rows: 4885 | states: 50


,state,naics,export_value_canada,st_fips,export_value_world,export_value,canada_share
0,Alabama,1112,596067.0,01,596067.0,596067.0,1.000000
1,Alabama,1119,2196319.0,01,44329361.0,2196319.0,0.049545
2,Alabama,1123,32396617.0,01,232875669.0,32396617.0,0.139116
3,Alabama,1129,280157.0,01,2095003.0,280157.0,0.133726
4,Alabama,1132,9888.0,01,61665.0,9888.0,0.160350


# STEP 3 — QCEW county employment in tariffed NAICS4 (weighted)

Mirrors Canadian cells 15–17 (establishment/employee exposure), but reads the 4,451 county QCEW
files, filters to private NAICS4 (agglvl 76, own 5), drops disclosure-suppressed cells, and keeps
employment in the tariffed NAICS set. No DA rollup — county is the unit.

In [26]:
# Map state NAME -> FIPS prefix so county employment can inherit its state's export weight
STATE_FIPS = {
 'Alabama':'01','Alaska':'02','Arizona':'04','Arkansas':'05','California':'06','Colorado':'08',
 'Connecticut':'09','Delaware':'10','District of Columbia':'11','Florida':'12','Georgia':'13',
 'Hawaii':'15','Idaho':'16','Illinois':'17','Indiana':'18','Iowa':'19','Kansas':'20','Kentucky':'21',
 'Louisiana':'22','Maine':'23','Maryland':'24','Massachusetts':'25','Michigan':'26','Minnesota':'27',
 'Mississippi':'28','Missouri':'29','Montana':'30','Nebraska':'31','Nevada':'32','New Hampshire':'33',
 'New Jersey':'34','New Mexico':'35','New York':'36','North Carolina':'37','North Dakota':'38',
 'Ohio':'39','Oklahoma':'40','Oregon':'41','Pennsylvania':'42','Rhode Island':'44',
 'South Carolina':'45','South Dakota':'46','Tennessee':'47','Texas':'48','Utah':'49','Vermont':'50',
 'Virginia':'51','Washington':'53','West Virginia':'54','Wisconsin':'55','Wyoming':'56'}
canada_exp['st_fips'] = canada_exp['state'].map(STATE_FIPS)
canada_exp = canada_exp.dropna(subset=['st_fips'])


In [27]:
# Read all county QCEW files, keep private NAICS4 (agglvl 76) non-suppressed rows
files = glob.glob(os.path.join(QCEW_DIR, '*.csv'))
print(f'{len(files)} QCEW area files')

keep = []
for i, f in enumerate(files):
    d = pd.read_csv(f, dtype={'area_fips':str,'industry_code':str,'disclosure_code':str},
                    usecols=['area_fips','own_code','industry_code','agglvl_code',
                             'disclosure_code','annual_avg_emplvl','annual_avg_estabs_count'])
    d = d[(d['agglvl_code']==QCEW_AGGLVL) & (d['own_code']==QCEW_OWN)]
    d = d[d['disclosure_code'].isna()]                 # drop suppressed
    if len(d): keep.append(d)
    if (i+1) % 500 == 0: print(f'  {i+1}/{len(files)}')

qcew = pd.concat(keep, ignore_index=True)
qcew['naics'] = qcew['industry_code'].str.replace(r'\D','',regex=True).str[:NAICS_DEPTH]
qcew['st_fips'] = qcew['area_fips'].str[:2]
print('county x naics4 private rows:', len(qcew), '| counties:', qcew['area_fips'].nunique())


4451 QCEW area files
  500/4451
  1000/4451
  1500/4451
  2000/4451
  2500/4451
  3000/4451
  3500/4451
  4000/4451
county x naics4 private rows: 175852 | counties: 3205


In [28]:
# County totals (all private NAICS4) — denominator for exposure %
county_total = (qcew.groupby('area_fips', as_index=False)
                .agg(all_emp=('annual_avg_emplvl','sum'),
                     all_estabs=('annual_avg_estabs_count','sum')))

def exposure(naics_set, label):
    sub = qcew[qcew['naics'].isin(naics_set)].copy()
    # attach state export weight (Canada export $ for that state x naics)
    sub = sub.merge(canada_exp[['st_fips','naics','canada_share']], on=['st_fips','naics'], how='left')
    # weighted exposed employment: employment scaled by Canada-export intensity proxy.
    # v1 (no world denominator): use presence-weighting — exposed emp = emp where the industry
    # exports to Canada at all; multiply by a normalized weight if/when world denominator added.
    sub['canada_share'] = sub['canada_share'].fillna(0) # states w/ no exports = 0
    # normalize weight to [0,1] within naics so it scales emp without exploding units (placeholder
    # For weighted exposure, use the actual share (0-1) as the weight
    sub['emp_weighted'] = sub['annual_avg_emplvl'] * sub['canada_share']

    # Then aggregate using sum of weighted employment instead of raw employment
    g = (sub.groupby('area_fips', as_index=False)
        .agg(**{f'{label}_emp':('annual_avg_emplvl','sum'),  # unweighted (for reference)
                f'{label}_emp_w':('emp_weighted','sum'),     # weighted by Canada export share
                f'{label}_estabs':('annual_avg_estabs_count','sum')}))
    return g

exp_pre  = exposure(naics_pre,  'pre')
exp_post = exposure(naics_post, 'post')
print('counties with pre exposure:', len(exp_pre), '| post:', len(exp_post))


counties with pre exposure: 2217 | post: 1352


In [29]:
# Assemble county exposure table with pre/post + percentages
county = (county_total
          .merge(exp_pre,  on='area_fips', how='left')
          .merge(exp_post, on='area_fips', how='left')
          .fillna(0))
for p in ['pre','post']:
    county[f'{p}_emp_pct'] = np.where(county['all_emp']>0, county[f'{p}_emp']/county['all_emp'], 0)
county['removed_emp']     = county['pre_emp']     - county['post_emp']
county['removed_emp_pct'] = county['pre_emp_pct'] - county['post_emp_pct']
print(county[['area_fips','all_emp','pre_emp','post_emp','pre_emp_pct','post_emp_pct','removed_emp_pct']].describe())
county.head()


            all_emp        pre_emp      post_emp  pre_emp_pct  post_emp_pct  \
count  3.205000e+03    3205.000000   3205.000000  3205.000000   3205.000000   
mean   3.620732e+04    2109.149766    517.186271     0.071333      0.013218   
std    1.460457e+05    9220.099599   2447.364983     0.128620      0.041416   
min    0.000000e+00       0.000000      0.000000     0.000000      0.000000   
25%    5.970000e+02       0.000000      0.000000     0.000000      0.000000   
50%    3.006000e+03      92.000000      0.000000     0.026224      0.000000   
75%    1.494500e+04     789.000000    175.000000     0.082847      0.010745   
max    3.910881e+06  264322.000000  57553.000000     1.000000      0.928771   

       removed_emp_pct  
count      3205.000000  
mean          0.058114  
std           0.121209  
min           0.000000  
25%           0.000000  
50%           0.015288  
75%           0.059102  
max           1.000000  


,area_fips,all_emp,all_estabs,pre_emp,pre_emp_w,pre_estabs,post_emp,post_emp_w,post_estabs,pre_emp_pct,post_emp_pct,removed_emp,removed_emp_pct
0,01001,5697,826,74.0,15.306540,10.0,36.0,0.549138,4.0,0.012989,0.006319,38.0,0.006670
1,01003,68462,8853,2257.0,371.747827,223.0,834.0,158.122356,61.0,0.032967,0.012182,1423.0,0.020785
2,01005,2036,219,36.0,1.783637,7.0,0.0,0.000000,0.0,0.017682,0.000000,36.0,0.017682
3,01007,637,99,50.0,0.762692,5.0,50.0,0.762692,5.0,0.078493,0.078493,0.0,0.000000
4,01009,3719,567,261.0,52.720843,26.0,200.0,42.914394,14.0,0.070180,0.053778,61.0,0.016402


# STEP 4 — Overlay 2024 county vote margin (the Brookings move)

No Canadian equivalent. Two-party margin from MIT Election Lab; >0 = Trump-won county.

In [30]:
v = pd.read_csv(COUNTY_VOTE)
v = v[(v['year']==2024) & (v['office']=='US PRESIDENT')].copy()
v['area_fips'] = v['county_fips'].astype('Int64').astype(str).str.zfill(5)
piv = (v[v['party'].isin(['REPUBLICAN','DEMOCRAT'])]
       .pivot_table(index='area_fips', columns='party', values='candidatevotes', aggfunc='sum')
       .reset_index())
piv['two_party'] = piv['REPUBLICAN'] + piv['DEMOCRAT']
piv['rep_margin'] = np.where(piv['two_party']>0,
                             (piv['REPUBLICAN']-piv['DEMOCRAT'])/piv['two_party'], np.nan)
vote = piv[['area_fips','rep_margin']]
county = county.merge(vote, on='area_fips', how='left')
print('counties with vote margin:', county['rep_margin'].notna().sum())
# The headline Brookings-style read: do high-exposure counties skew Trump?
hi = county[county['pre_emp_pct'] > county['pre_emp_pct'].quantile(0.9)]
print('mean rep_margin, top-decile exposed counties:', round(hi['rep_margin'].mean(),3))
print('mean rep_margin, all counties:', round(county['rep_margin'].mean(),3))


counties with vote margin: 3037
mean rep_margin, top-decile exposed counties: 0.421
mean rep_margin, all counties: 0.354


# STEP 5 — Attach county geometry, write GeoJSON + CSV

In [33]:
# County boundaries — use Census cartographic counties (TODO: set path)
# e.g. cb_2023_us_county_500k.shp ; join on GEOID == area_fips
COUNTY_SHP = '../raw/cb_2018_us_county_500k/cb_2018_us_county_500k.shp'   # TODO download
geo = gpd.read_file(COUNTY_SHP)
geo['area_fips'] = geo['GEOID'].astype(str).str.zfill(5)
geo = geo[['area_fips','NAME','geometry']]

out = geo.merge(county, on='area_fips', how='left')
out = out.to_crs('EPSG:4326')
out.to_file(f'{OUT_DIR}/geojson/county_exposure.geojson', driver='GeoJSON')
out.drop(columns='geometry').to_csv(f'{OUT_DIR}/csv/county_exposure.csv', index=False)
print('wrote county_exposure.geojson /.csv  —', len(out), 'counties')


wrote county_exposure.geojson /.csv  — 3233 counties


# STEP 6 (case studies) — NAICS6 cross-border pairs

For Detroit⇄Windsor (auto), Sault Ste. Marie (steel), etc., re-read QCEW at agglvl 78 (NAICS6)
for the specific county FIPS, since case studies want detail and aren't constrained by the
NAICS4 export ceiling. Hand-curated; see project notes for the pair list and sources.

In [34]:
# Example: pull NAICS6 auto employment for Wayne County MI (Detroit, FIPS 26163)
def naics6_for_county(fips):
    f = glob.glob(os.path.join(QCEW_DIR, f'*{fips}*.csv'))
    if not f: return None
    d = pd.read_csv(f[0], dtype={'industry_code':str,'disclosure_code':str})
    d = d[(d['agglvl_code']==78) & (d['own_code']==5) & (d['disclosure_code'].isna())]
    return d[['industry_code','industry_title','annual_avg_emplvl','annual_avg_estabs_count']]

# detroit = naics6_for_county('26163')   # uncomment once paths set
# auto NAICS6: 336111 (autos), 336390 (parts), 336370 (stamping), etc.
